# Sales Forecast Regression

**Tabular Regression Project**

Models: CatBoost · LightGBM · XGBoost
Baselines: LazyPredict · FLAML AutoML
Target: `sales`

## 2 · Project Overview

We predict **daily store-level sales** based on store ID, product category, day/month, weekend/holiday flags, advertising spend, discount percentage, temperature, and foot traffic.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a tabular regression dataset.
2. Encode categorical features for tree-based and linear models.
3. Build a baseline Linear Regression model.
4. Use LazyPredict for rapid model benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, XGBoost with GPU→CPU fallback.
7. Evaluate with RMSE, MAE, R², and residual analysis.

## 4 · Problem Statement

Given a store's daily context — product category, day of week, month, advertising spend, discounts, weather, and foot traffic — predict the daily sales amount.

## 5 · Why This Project Matters

- **Demand forecasting** drives inventory, staffing, and marketing allocation.
- Understanding sales drivers optimizes advertising ROI.
- Seasonal patterns require proper feature engineering.
- Demonstrates regression with temporal and marketing features.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 8,000 |
| **Features** | store_id, product_category, day_of_week, month, is_weekend, is_holiday, advertising_spend, discount_pct, temperature_f, foot_traffic |
| **Target** | sales (continuous, USD) |
| **Range** | ~$50 – $50K |

## 7 · Dataset Source and License Notes

- **Source**: Synthetic dataset generated for educational purposes.
- **License**: Public / educational use. No PII.
- **Limitations**: Simplified patterns — real-world data would have more features and noise.

## 8 · Environment Setup

Install any missing packages. GPU is auto-detected with CPU fallback.

In [1]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, root_mean_squared_error

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

Imports complete.


## 10 · Configuration / Constants

In [3]:
TARGET = "sales"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Target: {TARGET}")
print(f"Save dir: {SAVE_DIR}")

Target: sales
Save dir: E:\Github\Machine-Learning-Projects\Regression\Sales Forecast Regression


## 11 · Dataset Download or Loading

Synthetic dataset: 8,000 daily store-product sales records with marketing and store features.

In [4]:
np.random.seed(SEED)
n = 8000
store_id = np.random.randint(1, 11, n)
product_category = np.random.choice(["Electronics", "Clothing", "Food", "Home", "Sports"], n,
                                     p=[0.2, 0.25, 0.2, 0.2, 0.15])
day_of_week = np.random.randint(0, 7, n)
month = np.random.randint(1, 13, n)
is_weekend = (day_of_week >= 5).astype(int)
is_holiday = np.random.binomial(1, 0.05, n)
advertising_spend = np.round(np.random.lognormal(6, 1, n).clip(50, 50000), 2)
discount_pct = np.round(np.random.uniform(0, 40, n), 1)
temperature_f = np.round(np.random.normal(65, 15, n).clip(10, 110), 1)
foot_traffic = np.random.poisson(200, n).clip(20, 1000)

cat_mult = {"Electronics": 1.3, "Clothing": 1.0, "Food": 0.8, "Home": 0.9, "Sports": 1.1}
cat_val = np.array([cat_mult[c] for c in product_category])

seasonality = 1 + 0.15 * np.sin(2 * np.pi * (month - 1) / 12)

sales = (0.05 * advertising_spend + 10 * discount_pct + 0.5 * foot_traffic
         + 200 * is_weekend + 500 * is_holiday + 50 * store_id)
sales = sales * cat_val * seasonality
sales = np.round(sales + np.random.normal(0, 200, n), 2).clip(50, 50000)

df = pd.DataFrame({"store_id": store_id, "product_category": product_category,
                    "day_of_week": day_of_week, "month": month, "is_weekend": is_weekend,
                    "is_holiday": is_holiday, "advertising_spend": advertising_spend,
                    "discount_pct": discount_pct, "temperature_f": temperature_f,
                    "foot_traffic": foot_traffic, "sales": sales})
print(f"Dataset shape: {df.shape}")
print(f"Target stats:\n{df['sales'].describe()}")
df.head()

Dataset shape: (8000, 11)
Target stats:
count    8000.000000
mean      700.274046
std       336.995394
min        50.000000
25%       460.860000
50%       675.775000
75%       909.352500
max      2426.540000
Name: sales, dtype: float64


,store_id,product_category,day_of_week,month,is_weekend,is_holiday,advertising_spend,discount_pct,temperature_f,foot_traffic,sales
0,7,Clothing,2,6,0,0,1417.82,7.3,80.2,175,609.67
1,4,Home,6,9,1,0,189.09,37.8,62.8,181,667.57
2,8,Sports,0,4,0,0,369.13,17.2,61.8,192,803.27
3,5,Food,2,6,0,0,457.00,1.9,77.1,212,336.18
4,7,Food,0,1,0,1,379.14,28.5,75.1,215,770.75


## 12 · Data Validation Checks

Verify the dataset is complete and correctly structured.

In [5]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")
print(f"\nTarget stats:\n{df[TARGET].describe()}")

DATA VALIDATION

Shape: (8000, 11)

Missing values:
store_id             0
product_category     0
day_of_week          0
month                0
is_weekend           0
is_holiday           0
advertising_spend    0
discount_pct         0
temperature_f        0
foot_traffic         0
sales                0
dtype: int64

Duplicate rows: 0

Dtypes:
store_id               int32
product_category      object
day_of_week            int32
month                  int32
is_weekend             int64
is_holiday             int32
advertising_spend    float64
discount_pct         float64
temperature_f        float64
foot_traffic           int32
sales                float64
dtype: object

Target 'sales' confirmed.

Target stats:
count    8000.000000
mean      700.274046
std       336.995394
min        50.000000
25%       460.860000
50%       675.775000
75%       909.352500
max      2426.540000
Name: sales, dtype: float64


## 13 · Exploratory Data Analysis

Explore feature distributions, correlations, and relationships.

In [6]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes[0][0].scatter(df["advertising_spend"], df[TARGET], alpha=0.2, s=10)
axes[0][0].set_xlabel("Ad Spend"); axes[0][0].set_ylabel("Sales"); axes[0][0].set_title("Ad Spend vs Sales")

axes[0][1].scatter(df["foot_traffic"], df[TARGET], alpha=0.2, s=10)
axes[0][1].set_xlabel("Foot Traffic"); axes[0][1].set_ylabel("Sales"); axes[0][1].set_title("Foot Traffic vs Sales")

df.boxplot(column=TARGET, by="product_category", ax=axes[0][2])
axes[0][2].set_title("Sales by Category")
axes[0][2].tick_params(axis="x", rotation=45)

df.groupby("month")[TARGET].mean().plot(kind="bar", ax=axes[1][0], color="steelblue", edgecolor="black")
axes[1][0].set_title("Avg Sales by Month")

df.groupby("day_of_week")[TARGET].mean().plot(kind="bar", ax=axes[1][1], color="steelblue", edgecolor="black")
axes[1][1].set_title("Avg Sales by Day of Week")

corr = df.select_dtypes(include="number").corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[1][2], square=True, cbar_kws={"shrink":0.8})
axes[1][2].set_title("Correlation")

plt.tight_layout()
plt.show()

## 14 · Target Analysis

Examine the distribution of `sales`.

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df[TARGET], bins=30, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].set_title(f"Distribution of {TARGET}")
axes[0].set_xlabel("Sales ($)")

df.boxplot(column=TARGET, by="is_weekend", ax=axes[1])
axes[1].set_title("Sales: Weekday vs Weekend")

plt.tight_layout()
plt.show()
print(f"Range: ${df[TARGET].min():,.0f} to ${df[TARGET].max():,.0f}")
print(f"Mean: ${df[TARGET].mean():,.0f}, Median: ${df[TARGET].median():,.0f}")
print(f"Weekend premium: ${df[df['is_weekend']==1][TARGET].mean() - df[df['is_weekend']==0][TARGET].mean():,.0f}")

Range: $50 to $2,427
Mean: $700, Median: $676
Weekend premium: $199


## 15 · Train/Validation/Test Split Strategy

80/20 train/test split.

In [8]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Encode categorical features
cat_cols = X.select_dtypes(include="object").columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X[cat_cols] = oe.fit_transform(X[cat_cols])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Target — Train mean: {y_train.mean():,.2f}, Test mean: {y_test.mean():,.2f}")

Train: (6400, 10), Test: (1600, 10)
Target — Train mean: 699.49, Test mean: 703.40


## 16 · Preprocessing Strategy

- **Missing values**: None.
- **Categorical**: `product_category` encoded with OrdinalEncoder.
- **Scaling**: Not needed for tree models.
- **Temporal**: `day_of_week` and `month` kept as integers (tree models handle ordinal features).

## 17 · Feature Engineering

Sanitize column names and add engineered features.

In [9]:
X_train = X_train.copy()
X_test = X_test.copy()

X_train["ad_per_traffic"] = X_train["advertising_spend"] / (X_train["foot_traffic"] + 1)
X_test["ad_per_traffic"] = X_test["advertising_spend"] / (X_test["foot_traffic"] + 1)

X_train["month_sin"] = np.sin(2 * np.pi * X_train["month"] / 12)
X_test["month_sin"] = np.sin(2 * np.pi * X_test["month"] / 12)

X_train["month_cos"] = np.cos(2 * np.pi * X_train["month"] / 12)
X_test["month_cos"] = np.cos(2 * np.pi * X_test["month"] / 12)

clean = [c.replace(" ", "_").replace("-", "_").replace(".", "_") for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

Features (13): ['store_id', 'product_category', 'day_of_week', 'month', 'is_weekend', 'is_holiday', 'advertising_spend', 'discount_pct', 'temperature_f', 'foot_traffic', 'ad_per_traffic', 'month_sin', 'month_cos']


## 18 · Baseline Model

Linear Regression as our baseline regressor.

In [10]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

rmse_bl = root_mean_squared_error(y_test, y_pred_bl)
mae_bl = mean_absolute_error(y_test, y_pred_bl)
r2_bl = r2_score(y_test, y_pred_bl)

print("=" * 50)
print("BASELINE: Linear Regression")
print("=" * 50)
print(f"RMSE : {rmse_bl:,.2f}")
print(f"MAE  : {mae_bl:,.2f}")
print(f"R2   : {r2_bl:.4f}")

BASELINE: Linear Regression
RMSE : 232.46
MAE  : 186.89
R2   : 0.5525


## 19 · LazyPredict Benchmark

Fit dozens of regressors in one call for a quick ranking.

In [11]:
from lazypredict.Supervised import LazyRegressor

lazy = LazyRegressor(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 15 Regressors:")
print(lazy_models.head(15).to_string())

LazyPredict — Top 15 Regressors:
                               Adjusted R-Squared  R-Squared        RMSE  Time Taken
Model                                                                               
GradientBoostingRegressor                0.660339   0.663101  201.694912    0.955311
HistGradientBoostingRegressor            0.660238   0.663000  201.724980    0.325584
LGBMRegressor                            0.658186   0.660965  202.333239    0.077266
CatBoostRegressor                        0.655891   0.658689  203.011318    4.089503
RandomForestRegressor                    0.623911   0.626968  212.235432    2.628886
ExtraTreesRegressor                      0.622162   0.625234  212.728142    1.830148
XGBRegressor                             0.591382   0.594704  221.223403    0.180218
BaggingRegressor                         0.581764   0.585165  223.811750    0.334698
AdaBoostRegressor                        0.576761   0.580202  225.146394    0.441307
MLPRegressor                    

## 20 · FLAML AutoML Run

Automated model selection and hyperparameter tuning (60s budget).

In [12]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(X_train, y_train, task="regression", time_budget=60,
                 estimator_list=["lgbm", "rf", "extra_tree", "catboost"],
                 verbose=0, seed=SEED)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best model: {flaml_automl.best_estimator}")
print(f"RMSE: {root_mean_squared_error(y_test, y_pred_flaml):,.2f}")
print(f"R2  : {r2_score(y_test, y_pred_flaml):.4f}")

FLAML best model: catboost
RMSE: 201.27
R2  : 0.6645


## 21 · Additional Models: CatBoost, LightGBM, XGBoost

Train the modern tabular model stack. GPU auto-detected with CPU fallback.

In [13]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# CatBoost
try:
    from catboost import CatBoostRegressor
    t0 = time.perf_counter()
    try:
        cb = CatBoostRegressor(iterations=300, learning_rate=0.05, depth=6,
                                task_type="GPU", devices="0", verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    except Exception:
        cb = CatBoostRegressor(iterations=300, learning_rate=0.05, depth=6,
                                verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test)
    print(f"CatBoost RMSE: {root_mean_squared_error(y_test, results['CatBoost']):,.2f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost failed: {e}")
gpu_cleanup()

# LightGBM
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    try:
        lg = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    except Exception:
        lg = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM RMSE: {root_mean_squared_error(y_test, results['LightGBM']):,.2f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM failed: {e}")
gpu_cleanup()

# XGBoost
try:
    from xgboost import XGBRegressor
    t0 = time.perf_counter()
    try:
        xgb_model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  device="cuda", tree_method="hist", verbosity=0,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    except Exception:
        xgb_model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  tree_method="hist", verbosity=0,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost RMSE: {root_mean_squared_error(y_test, results['XGBoost']):,.2f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost failed: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["Linear Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

CatBoost RMSE: 199.53  (2.5s)


Training until validation scores don't improve for 30 rounds


Early stopping, best iteration is:
[168]	valid_0's l2: 40041.9
LightGBM RMSE: 200.10  (1.6s)


XGBoost failed: XGBModel.fit() got an unexpected keyword argument 'early_stopping_rounds'


## 22 · Top 3 Model Selection

Rank all models by RMSE and select the top 3.

In [14]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "RMSE": round(root_mean_squared_error(y_test, yp), 2),
        "MAE": round(mean_absolute_error(y_test, yp), 2),
        "R2": round(r2_score(y_test, yp), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("RMSE")
print("All Model Rankings (by RMSE):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

All Model Rankings (by RMSE):
                     RMSE     MAE      R2
CatBoost           199.53  158.17  0.6703
LightGBM           200.10  158.63  0.6684
FLAML              201.27  159.71  0.6645
Linear Regression  232.46  186.89  0.5525

Top 3 models: ['CatBoost', 'LightGBM', 'FLAML']


## 23 · Final Training and Evaluation of Top 3

Detailed metrics and predicted-vs-actual plots.

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    rmse = root_mean_squared_error(y_test, yp)
    r2 = r2_score(y_test, yp)

    axes[i].scatter(y_test, yp, alpha=0.6, s=40, edgecolors="black")
    mn, mx = min(y_test.min(), yp.min()), max(y_test.max(), yp.max())
    axes[i].plot([mn, mx], [mn, mx], "r--", lw=2)
    axes[i].set_title(f"{name}\nRMSE={rmse:,.2f}  R2={r2:.4f}")
    axes[i].set_xlabel("Actual")
    axes[i].set_ylabel("Predicted")

plt.suptitle("Top 3 Models — Predicted vs Actual", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "top3_predicted_vs_actual.png"), dpi=120, bbox_inches="tight")
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    print(f"\n  {name}:")
    print(f"    RMSE : {root_mean_squared_error(y_test, yp):,.2f}")
    print(f"    MAE  : {mean_absolute_error(y_test, yp):,.2f}")
    print(f"    R2   : {r2_score(y_test, yp):.4f}")


Detailed Metrics — Top 3:

  CatBoost:
    RMSE : 199.53
    MAE  : 158.17
    R2   : 0.6703

  LightGBM:
    RMSE : 200.10
    MAE  : 158.63
    R2   : 0.6684

  FLAML:
    RMSE : 201.27
    MAE  : 159.71
    R2   : 0.6645


## 24 · Error Analysis

Examine residuals from the best model.

In [16]:
best_name = top3_names[0]
best_pred = results[best_name]
residuals = y_test.values - best_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(residuals, bins=20, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].axvline(0, color="red", linestyle="--")
axes[0].set_title(f"{best_name} — Residual Distribution")
axes[0].set_xlabel("Residual (Actual - Predicted)")

axes[1].scatter(best_pred, residuals, alpha=0.6, s=40, edgecolors="black")
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_title(f"{best_name} — Residual vs Predicted")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Residual")

sorted_idx = np.argsort(best_pred)
axes[2].scatter(best_pred[sorted_idx], np.abs(residuals[sorted_idx]), alpha=0.6, s=40, edgecolors="black")
axes[2].set_title(f"{best_name} — |Residual| vs Predicted")
axes[2].set_xlabel("Predicted"); axes[2].set_ylabel("|Residual|")

plt.tight_layout()
plt.show()

print(f"Residual stats ({best_name}):")
print(f"  Mean: {residuals.mean():,.2f}, Std: {residuals.std():,.2f}")
print(f"  Min: {residuals.min():,.2f}, Max: {residuals.max():,.2f}")
print(f"  Median: {np.median(residuals):,.2f}")

Residual stats (CatBoost):
  Mean: -1.61, Std: 199.52
  Min: -697.52, Max: 679.60
  Median: -1.78


## 25 · Interpretation and Business Insight

**Key findings:**
- **Foot traffic** is the strongest direct sales predictor.
- **Advertising spend** has a diminishing-returns (logarithmic) effect.
- **Weekends and holidays** generate significant sales lifts.
- **Seasonality** (monthly cyclic pattern) affects all categories.
- **Electronics** has the highest sales per transaction.

**Business takeaway:** Maximize foot traffic and time promotions around weekends/holidays. Ad spend shows diminishing returns beyond ~$10K.

## 26 · Limitations

1. Synthetic data with simplified sales dynamics.
2. No competitor actions or pricing.
3. Missing product-level granularity (only category).
4. No inventory constraints (stockouts).
5. Temperature effect is minimal in this dataset.

## 27 · How to Improve This Project

1. Use real POS data with product-level detail.
2. Add competitor pricing and promotions.
3. Include inventory levels (stockout modeling).
4. Use time-series models for temporal patterns.
5. Add online/omnichannel sales data.

## 28 · Production Considerations

- Deploy for daily demand forecasting.
- Integrate with inventory replenishment systems.
- Monitor forecast accuracy by store and category.
- Update weekly with new sales data.
- Combine with promotion planning tools.

## 29 · Common Mistakes

1. Ignoring seasonality in cross-validation (use time-aware splits).
2. Not log-transforming advertising spend (diminishing returns).
3. Treating store_id as continuous instead of categorical.
4. Not accounting for stockouts biasing sales data.
5. Using future data in features (leakage).

## 30 · Mini Challenge / Exercises

1. Log-transform `advertising_spend` and compare results.
2. Remove `foot_traffic` — how much does RMSE increase?
3. Add cyclical encoding for `day_of_week` (sin/cos).
4. Build separate models by product category.
5. Plot the marginal return on advertising spend.

## 31 · Final Summary / Key Takeaways

1. **Foot traffic** is the strongest direct sales driver.
2. **Advertising** shows diminishing returns (logarithmic).
3. **Weekends/holidays** provide significant sales lifts.
4. **Seasonality** requires proper cyclical encoding.
5. **Category** matters — Electronics generates highest revenue.
6. **Time-series models** would better capture temporal patterns.

## Save Metrics

In [17]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "rmse": round(float(root_mean_squared_error(y_test, yp)), 2),
        "mae": round(float(mean_absolute_error(y_test, yp)), 2),
        "r2": round(float(r2_score(y_test, yp)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))

Metrics saved to E:\Github\Machine-Learning-Projects\Regression\Sales Forecast Regression\metrics.json
{
  "CatBoost": {
    "rmse": 199.53,
    "mae": 158.17,
    "r2": 0.6703
  },
  "LightGBM": {
    "rmse": 200.1,
    "mae": 158.63,
    "r2": 0.6684
  },
  "Linear Regression": {
    "rmse": 232.46,
    "mae": 186.89,
    "r2": 0.5525
  },
  "FLAML": {
    "rmse": 201.27,
    "mae": 159.71,
    "r2": 0.6645
  }
}
